<a href="https://colab.research.google.com/github/mdkamrulhasan/data_mining_kdd/blob/main/notebooks/Diabetes_EDA_Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LOADING LIBRARIES**

In [ ]:
#Loading libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# **DATA IMPORTATION**

In [ ]:
diabetes = pd.read_csv("https://raw.githubusercontent.com/mdkamrulhasan/data_mining_kdd/main/data/diabetes/diabetes_data.csv", encoding='unicode_escape')

# **EXPLORATORY DATA ANALYSIS(EDA)**

In [ ]:
#nspecting the first five rows of the dataset

diabetes.head()

In [ ]:
diabetes.info()

In [ ]:
#converting variables to correct datatypes

diabetes['Age'] = diabetes['Age'].astype(int)
diabetes['Sex'] = diabetes['Sex'].astype(int)
diabetes['HighChol'] = diabetes['HighChol'].astype(int)
diabetes['CholCheck'] = diabetes['CholCheck'].astype(int)
diabetes['Smoker'] = diabetes['Smoker'].astype(int)
diabetes['HeartDiseaseorAttack'] = diabetes['HeartDiseaseorAttack'].astype(int)
diabetes['PhysActivity'] = diabetes['PhysActivity'].astype(int)
diabetes['Fruits'] = diabetes['Fruits'].astype(int)
diabetes['Veggies'] = diabetes['Veggies'].astype(int)
diabetes['HvyAlcoholConsump'] = diabetes['HvyAlcoholConsump'].astype(int)
diabetes['GenHlth'] = diabetes['GenHlth'].astype(int)
diabetes['MentHlth'] = diabetes['MentHlth'].astype(int)
diabetes['PhysHlth'] = diabetes['PhysHlth'].astype(int)
diabetes['DiffWalk'] = diabetes['DiffWalk'].astype(int)
diabetes['Stroke'] = diabetes['Stroke'].astype(int)
diabetes['HighBP'] = diabetes['HighBP'].astype(int)
diabetes['Diabetes'] = diabetes['Diabetes'].astype(int)


# **FEATURE ENGINEERING**

# **CORRELATION ANALYSIS**

In [ ]:
# Correlation studies

plt.figure(figsize = (13,9))
sns.heatmap(diabetes.corr(), annot = True, cmap = 'coolwarm', fmt = '.2f')
plt.title('correlation matrix')
plt.show()

In [ ]:
#Feature correlation with target variable

diabetes.corr()['Diabetes'].sort_values(ascending=False).plot(kind = 'bar', grid = True, figsize = (15, 6))

# **HYPOTHESIS TESTING**

In [ ]:
diabetes.info()

In [ ]:
diabetes = diabetes.drop(columns=['BMI_Category'])

In [ ]:
import scipy.stats as stats

# variables excluding 'Diabetes'
variables_to_test = diabetes.columns[diabetes.columns != 'Diabetes']

# Empty DataFrame to store the results
test_results = pd.DataFrame(columns=['Variable', 'Test_Statistic', 'P_Value'])

# Loop through each variable and perform the t-test
for variable in variables_to_test:
    group_0 = diabetes[diabetes['Diabetes'] == 0][variable]
    group_1 = diabetes[diabetes['Diabetes'] == 1][variable]

    t_stat, p_value = stats.ttest_ind(group_0, group_1, equal_var=False)

    # Append results to the DataFrame
    test_results = test_results.append({
        'Variable': variable,
        'Test_Statistic': t_stat,
        'P_Value': p_value
    }, ignore_index=True)

# Display the results
print(test_results)


# **FEATURE IMPORTANCE**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#ExtractIng features and target variable
X = diabetes.drop('Diabetes', axis=1)
y = diabetes['Diabetes']

#Initialize a RandomForestClassifier
rf_classifier = RandomForestClassifier(random_state=42)

#Fit the model
rf_classifier.fit(X, y)

feature_importances = pd.Series(rf_classifier.feature_importances_, index=X.columns)
feature_importances.sort_values(ascending=False).plot(kind='barh', figsize=(12, 6))

plt.title('Feature Importances in Random Forest Classifier')
plt.xlabel('Importance')
plt.ylabel('Features')


In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

k = 20
chi2_selector = SelectKBest(chi2, k=k)
X_chi2 = chi2_selector.fit_transform(X, y)

#Getting chi-squared test statistics and p-values
chi2_stats = chi2_selector.scores_
p_values = chi2_selector.pvalues_

#DataFrame displaying chi-squared statistics and p-values for each feature
chi2_df = pd.DataFrame({'Feature': X.columns, 'Chi2_Stat': chi2_stats, 'P_Value': p_values})

# Sorting features by chi-squared statistics in descending order and p-values in ascending order
chi2_df = chi2_df.sort_values(by=['Chi2_Stat', 'P_Value'], ascending=[False, True])

# Chi-squared Test for Feature Selection
print(chi2_df['Feature'])

In [ ]:
#Setting up the figure
plt.figure(figsize=(10, 6))

# Plotting the features
plt.barh(chi2_df['Feature'], chi2_df['Chi2_Stat'])

#Labels and Title
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title('Chi-squared Test for Feature Selection')

# Show the plot
plt.show()